## 歩行モデル訓練用コード(Colabで学習)

In [ ]:
!pip install genesis-world tensordict

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os

WORKSPACE = Path("/content/drive/MyDrive/PhysicalAI_adv/dog")
os.makedirs(WORKSPACE, exist_ok=True)
os.chdir(WORKSPACE)

## ボールを持ってきてくれる犬ロボット

[使用するロボット]

Go2(頭のカメラ付き・シュミレーション)

[作戦]
1. Genesis worldで歩行モデルを訓練
2. CaPで歩行モデルに指示を送り、タスクを遂行
3. 必要ならCaP-XのCaP-RLでコード生成モデルを訓練

## ロボット歩行方策設計

**1. 入力**
- 現在の関節角度(全部)
- ~~Go2の頭カメラ~~
- 指示3つ
  - 前後の速度
  - 左右の速度
  - 旋回用の角速度

**2. 出力**
- 次の関節角度


**3. モデル**

1. ~~画像 - ViTに通して数十次元ベクトルに圧縮(Wi)~~
2. 関節角度 - 一列に並べる(Wj)
3. 指示 - 三次元(Wc)

Wj + 区切り + Wc → MLP → 次の関節角度


**4. 訓練方法**

PPOを使用

[報酬リスト]
- 前後左右の速度指示への追従
- 旋回指示への追従
- 機体がデフォルトに近いか
- 関節角度が急変しないか
- 上下の揺れが大きすぎないか

**非対称なActor-Criticによる学習**

[Actor]
- 入力2種類

[Critic]
- 入力2種類に加えて
- 現在速度
- 接触情報(これ取れるかな？)

Criticのネットワーク構成

1. ~~画像 - ViTに通して数十次元ベクトルに圧縮(Wi)~~
2. 関節角度・現在速度 - 一列に並べる(Wj)
3. 指示 - 三次元(Wc)
4. 接触情報 - ?

Wj + 区切り + Wc → MLP → 状態価値関数

## 学習環境作り

平面にボール(ぷにぷにのゴムボール的な)を散らばした環境

一定期間(割と細かめ)で指示を入れかえ
- 前進速度追従
- 緩やかなカーブ
- 左右に位置を調節しながら前進
- Uターン





In [ ]:
import math

import torch
from tensordict import TensorDict

import genesis as gs
from genesis.utils.geom import quat_to_xyz, transform_by_quat, inv_quat, transform_quat_by_quat


def gs_rand(lower, upper, batch_shape, zero_prob):
    assert lower.shape == upper.shape
    shape = (*batch_shape, *lower.shape)
    out = (upper - lower) * torch.rand(shape, dtype=gs.tc_float, device=gs.device) + lower
    zero = torch.rand((*batch_shape, 1), device=gs.device) < zero_prob
    return out.masked_fill(zero, 0.0)


class Go2Env:
    def __init__(self, num_envs, env_cfg, obs_cfg, reward_cfg, command_cfg, show_viewer=False):
        self.num_envs: int = num_envs
        self.num_actions = env_cfg["num_actions"]
        self.cfg = env_cfg
        self.num_commands = command_cfg["num_commands"]
        self.device = gs.device

        self.simulate_action_latency = True  # there is a 1 step latency on real robot
        self.dt = 0.02  # control frequency on real robot is 50hz
        self.max_episode_length = math.ceil(env_cfg["episode_length_s"] / self.dt)

        self.env_cfg = env_cfg
        self.obs_cfg = obs_cfg
        self.reward_cfg = reward_cfg
        self.command_cfg = command_cfg

        self.obs_scales: dict[str, float] = obs_cfg["obs_scales"]
        self.reward_scales: dict[str, float] = reward_cfg["reward_scales"]

        self.scene = gs.Scene(
            sim_options=gs.options.SimOptions(
                dt=self.dt,
                substeps=2,
            ),
            rigid_options=gs.options.RigidOptions(
                enable_self_collision=False,
                # For this locomotion policy, there are usually no more than 20 collision pairs. Setting a low value
                # can save memory. Violating this condition will raise an exception.
                max_collision_pairs=20,
                tolerance=1e-5,
            ),
        )

        # add plain
        self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/plane/plane.urdf",
                fixed=True,
            )
        )

        self.robot = self.scene.add_entity(
            gs.morphs.URDF(
                file="urdf/go2/urdf/go2.urdf",
                pos=self.env_cfg["base_init_pos"],
                quat=self.env_cfg["base_init_quat"],
            ),
        )

        self.scene.build(n_envs=num_envs)

        # names to indices
        self.motors_dof_idx = torch.tensor(
            [self.robot.get_joint(name).dof_start for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_int,
            device=gs.device,
        )
        self.actions_dof_idx = torch.argsort(self.motors_dof_idx)

        # PD control parameters
        self.robot.set_dofs_kp([self.env_cfg["kp"]] * self.num_actions, self.motors_dof_idx)
        self.robot.set_dofs_kv([self.env_cfg["kd"]] * self.num_actions, self.motors_dof_idx)

        # Define global gravity direction vector
        self.global_gravity = torch.tensor([0.0, 0.0, -1.0], dtype=gs.tc_float, device=gs.device)

        # Initial state
        self.init_base_pos = torch.tensor(self.env_cfg["base_init_pos"], dtype=gs.tc_float, device=gs.device)
        self.init_base_quat = torch.tensor(self.env_cfg["base_init_quat"], dtype=gs.tc_float, device=gs.device)
        self.inv_base_init_quat = inv_quat(self.init_base_quat)
        self.init_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][joint.name] for joint in self.robot.joints[1:]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.init_qpos = torch.concatenate((self.init_base_pos, self.init_base_quat, self.init_dof_pos))
        self.init_projected_gravity = transform_by_quat(self.global_gravity, self.inv_base_init_quat)

        # initialize buffers
        self.base_lin_vel = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_ang_vel = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.projected_gravity = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.rew_buf = torch.empty((self.num_envs,), dtype=gs.tc_float, device=gs.device)
        self.reset_buf = torch.ones((self.num_envs,), dtype=gs.tc_bool, device=gs.device)
        self.episode_length_buf = torch.empty((self.num_envs,), dtype=gs.tc_int, device=gs.device)
        self.commands = torch.empty((self.num_envs, self.num_commands), dtype=gs.tc_float, device=gs.device)
        self.commands_scale = torch.tensor(
            [self.obs_scales["lin_vel"], self.obs_scales["lin_vel"], self.obs_scales["ang_vel"]],
            device=gs.device,
            dtype=gs.tc_float,
        )
        self.commands_limits: tuple[torch.Tensor, torch.Tensor] = tuple(
            torch.tensor(values, dtype=gs.tc_float, device=gs.device)
            for values in zip(
                self.command_cfg["lin_vel_x_range"],
                self.command_cfg["lin_vel_y_range"],
                self.command_cfg["ang_vel_range"],
            )
        )
        self.actions = torch.zeros((self.num_envs, self.num_actions), dtype=gs.tc_float, device=gs.device)
        self.last_actions = torch.zeros_like(self.actions)
        self.dof_pos = torch.empty_like(self.actions)
        self.dof_vel = torch.empty_like(self.actions)
        self.last_dof_vel = torch.zeros_like(self.actions)
        self.base_pos = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.base_quat = torch.empty((self.num_envs, 4), dtype=gs.tc_float, device=gs.device)
        self.base_euler = torch.empty((self.num_envs, 3), dtype=gs.tc_float, device=gs.device)
        self.default_dof_pos = torch.tensor(
            [self.env_cfg["default_joint_angles"][name] for name in self.env_cfg["joint_names"]],
            dtype=gs.tc_float,
            device=gs.device,
        )
        self.extras = dict()  # extra information for logging

        # prepare reward functions and multiply reward scales by dt
        self.reward_functions, self.episode_sums = dict(), dict()
        for name in self.reward_scales.keys():
            self.reward_scales[name] *= self.dt
            self.reward_functions[name] = getattr(self, "_reward_" + name)
            self.episode_sums[name] = torch.zeros((self.num_envs,), dtype=gs.tc_float, device=gs.device)

        link_names = [link.name for link in self.robot.links]
        suffix = "foot" if "FL_foot" in link_names else "calf"
        foot_names = [f"{leg}_{suffix}" for leg in ("FL", "FR", "RL", "RR")]
        self.foot_ids = torch.tensor(
            [link_names.index(n) for n in foot_names], dtype=torch.long, device=gs.device
        )

        self.reset()

    def _resample_commands(self, envs_idx):
        commands = gs_rand(*self.commands_limits, (self.num_envs,), 0.1)
        if envs_idx is None:
            self.commands.copy_(commands)
        else:
            torch.where(envs_idx[:, None], commands, self.commands, out=self.commands)

    def step(self, actions):
        self.actions = torch.clip(actions, -self.env_cfg["clip_actions"], self.env_cfg["clip_actions"])
        exec_actions = self.last_actions if self.simulate_action_latency else self.actions
        target_dof_pos = exec_actions * self.env_cfg["action_scale"] + self.default_dof_pos
        self.robot.control_dofs_position(target_dof_pos[:, self.actions_dof_idx], slice(6, 18))
        self.scene.step()

        # update buffers
        self.episode_length_buf += 1
        self.base_pos = self.robot.get_pos()
        self.base_quat = self.robot.get_quat()
        self.base_euler = quat_to_xyz(
            transform_quat_by_quat(self.inv_base_init_quat, self.base_quat), rpy=True, degrees=True
        )
        inv_base_quat = inv_quat(self.base_quat)
        self.base_lin_vel = transform_by_quat(self.robot.get_vel(), inv_base_quat)
        self.base_ang_vel = transform_by_quat(self.robot.get_ang(), inv_base_quat)
        self.projected_gravity = transform_by_quat(self.global_gravity, inv_base_quat)
        self.dof_pos = self.robot.get_dofs_position(self.motors_dof_idx)
        self.dof_vel = self.robot.get_dofs_velocity(self.motors_dof_idx)

        # compute reward
        self.rew_buf.zero_()
        for name, reward_func in self.reward_functions.items():
            rew = reward_func() * self.reward_scales[name]
            self.rew_buf += rew
            self.episode_sums[name] += rew

        self._resample_commands(self.episode_length_buf % int(self.env_cfg["resampling_time_s"] / self.dt) == 0)

        # check termination and reset
        self.reset_buf = self.episode_length_buf > self.max_episode_length
        self.reset_buf |= torch.abs(self.base_euler[:, 1]) > self.env_cfg["termination_if_pitch_greater_than"]
        self.reset_buf |= torch.abs(self.base_euler[:, 0]) > self.env_cfg["termination_if_roll_greater_than"]
        self.reset_buf |= self.scene.rigid_solver.get_error_envs_mask()

        # Compute timeout
        self.extras["time_outs"] = (self.episode_length_buf > self.max_episode_length).to(dtype=gs.tc_float)

        # Reset environment if necessary
        self._reset_idx(self.reset_buf)

        # update observations
        self._update_observation()

        self.last_actions.copy_(self.actions)
        self.last_dof_vel.copy_(self.dof_vel)

        return self.get_observations(), self.rew_buf.clone(), self.reset_buf, self.extras

    def get_observations(self):
        return TensorDict({"policy": self.obs_buf, "critic": self.critic_obs}, batch_size=[self.num_envs])

    def _reset_idx(self, envs_idx=None):
        # reset state
        self.robot.set_qpos(self.init_qpos, envs_idx=envs_idx, zero_velocity=True, skip_forward=True)

        # reset buffers
        if envs_idx is None:
            self.base_pos.copy_(self.init_base_pos)
            self.base_quat.copy_(self.init_base_quat)
            self.projected_gravity.copy_(self.init_projected_gravity)
            self.dof_pos.copy_(self.init_dof_pos)
            self.base_lin_vel.zero_()
            self.base_ang_vel.zero_()
            self.dof_vel.zero_()
            self.actions.zero_()
            self.last_actions.zero_()
            self.last_dof_vel.zero_()
            self.episode_length_buf.zero_()
            self.reset_buf.fill_(True)
        else:
            torch.where(envs_idx[:, None], self.init_base_pos, self.base_pos, out=self.base_pos)
            torch.where(envs_idx[:, None], self.init_base_quat, self.base_quat, out=self.base_quat)
            torch.where(
                envs_idx[:, None], self.init_projected_gravity, self.projected_gravity, out=self.projected_gravity
            )
            torch.where(envs_idx[:, None], self.init_dof_pos, self.dof_pos, out=self.dof_pos)
            self.base_lin_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.base_ang_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_actions.masked_fill_(envs_idx[:, None], 0.0)
            self.last_dof_vel.masked_fill_(envs_idx[:, None], 0.0)
            self.episode_length_buf.masked_fill_(envs_idx, 0)
            self.reset_buf.masked_fill_(envs_idx, True)

        # fill extras
        n_envs = envs_idx.sum() if envs_idx is not None else self.num_envs
        self.extras["episode"] = {}
        for key, value in self.episode_sums.items():
            if envs_idx is None:
                mean = value.mean()
            else:
                mean = torch.where(n_envs > 0, value[envs_idx].sum() / n_envs, 0.0)
            self.extras["episode"]["rew_" + key] = mean / self.env_cfg["episode_length_s"]
            if envs_idx is None:
                value.zero_()
            else:
                value.masked_fill_(envs_idx, 0.0)

        # random sample command upon reset
        self._resample_commands(envs_idx)

    def _update_observation(self):
        obs_terms = [
            self.base_ang_vel * self.obs_scales["ang_vel"],  # 3
            self.projected_gravity,  # 3
            self.commands * self.commands_scale,  # 3
            (self.dof_pos - self.default_dof_pos) * self.obs_scales["dof_pos"],  # 12
            self.dof_vel * self.obs_scales["dof_vel"],  # 12
            self.actions,  # 12
        ]

        self.obs_buf = torch.cat(
            obs_terms,
            dim=-1,
        )
        foot_force = self.robot.get_links_net_contact_force()[:, self.foot_ids, :]
        foot_contact = (foot_force.norm(dim=-1) > 1.0).float()

        # Observations for Critic only
        self.critic_obs = torch.cat(
            [*obs_terms,
            self.base_lin_vel * self.obs_scales["lin_vel"],
            foot_contact],
            dim=-1
        )

    def reset(self):
        self._reset_idx()
        self._update_observation()
        return self.get_observations()

    # Reward functions.
    def _reward_tracking_lin_vel(self):
        # Tracking of linear velocity commands (xy axes)
        lin_vel_error = torch.sum(torch.square(self.commands[:, :2] - self.base_lin_vel[:, :2]), dim=1)
        return torch.exp(-lin_vel_error / self.reward_cfg["tracking_sigma"])

    def _reward_tracking_ang_vel(self):
        # Tracking of angular velocity commands (yaw)
        ang_vel_error = torch.square(self.commands[:, 2] - self.base_ang_vel[:, 2])
        return torch.exp(-ang_vel_error / self.reward_cfg["tracking_sigma"])

    def _reward_lin_vel_z(self):
        # Penalize z axis base linear velocity
        return torch.square(self.base_lin_vel[:, 2])

    def _reward_action_rate(self):
        # Penalize changes in actions
        return torch.sum(torch.square(self.last_actions - self.actions), dim=1)

    def _reward_similar_to_default(self):
        # Penalize joint poses far away from default pose
        return torch.sum(torch.abs(self.dof_pos - self.default_dof_pos), dim=1)

In [ ]:
## Models
import torch
from torch import nn

class Actor(nn.Module):
  def __init__(self, input_dim, output_dim):
    super().__init__()
    self.mean = nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.GELU(),
        nn.Linear(512, 256),
        nn.GELU(),
        nn.Linear(256, 128),
        nn.GELU(),
        nn.Linear(128, output_dim)
    )
    self.log_std = nn.Parameter(torch.full((output_dim,), 0.01 ))

  def forward(self, x):
    mean = self.mean(x)
    return torch.distributions.Normal(mean, torch.exp(self.log_std))

class Critic(nn.Module):
  def __init__(self, input_dim, output_dim=1):
    super().__init__()
    self.linear_gelu = nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.GELU(),
        nn.Linear(512,256),
        nn.GELU(),
        nn.Linear(256, 128),
        nn.GELU(),
        nn.Linear(128, output_dim)
    )

  def forward(self,x):
    return self.linear_gelu(x)

In [ ]:
## Trainer

import os, json
import genesis as gs

TRAINING_NAME = "thefirststep"
DATA_DIR = WORKSPACE / TRAINING_NAME
os.makedirs(DATA_DIR, exist_ok=True)
TRAIN_CONFIG = DATA_DIR / "config.json"

WEIGHTS = DATA_DIR / "weights"
os.makedirs(WEIGHTS, exist_ok=True)
ACTOR_WEIGHT = WEIGHTS / "actor.pth"
CRITIC_WEIGHT = WEIGHTS / "critic.pth"
ACTOR_OPTIMIZER_WEIGHT = WEIGHTS / "actor_optimizer.pth"
CRITIC_OPTIMIZER_WEIGHT = WEIGHTS / "critic_optimizer.pth"

TRAIN_STATUS = WEIGHTS / "train_status.json"

LOGS_DIR = DATA_DIR / "logs"

LINEAR_VELOCITY_X_RANGE = [-0.5, 1.0]
LINEAR_VELOCITY_Y_RANGE = [-0.5, 0.5]
ANGULAR_VELOCITY_RANGE = [-1.5, 1.5]

def train_cfgs():
  num_envs = 16384
  ppo_cfg = {
      "epsilon": 0.2,
      "gamma":0.99,
      "lam" : 0.95,
      "entropy_beta" : 0.005,
      "num_epochs":4
  }
  trainer_cfg = {
      "step_per_iter" : 24,
      "iter" : 3000,
      "actor_adam_lr" : 3e-4,
      "critic_adam_lr" : 3e-4,
      "num_mini_batches":16
  }
  save_cfg={
      "save_iter":100,
  }
  return num_envs, ppo_cfg, trainer_cfg, save_cfg

def get_cfgs():
    env_cfg = {
        "num_actions": 12,
        # joint/link names
        "default_joint_angles": {  # [rad]
            "FL_hip_joint": 0.0,
            "FR_hip_joint": 0.0,
            "RL_hip_joint": 0.0,
            "RR_hip_joint": 0.0,
            "FL_thigh_joint": 0.8,
            "FR_thigh_joint": 0.8,
            "RL_thigh_joint": 1.0,
            "RR_thigh_joint": 1.0,
            "FL_calf_joint": -1.5,
            "FR_calf_joint": -1.5,
            "RL_calf_joint": -1.5,
            "RR_calf_joint": -1.5,
        },
        "joint_names": [
            "FR_hip_joint",
            "FR_thigh_joint",
            "FR_calf_joint",
            "FL_hip_joint",
            "FL_thigh_joint",
            "FL_calf_joint",
            "RR_hip_joint",
            "RR_thigh_joint",
            "RR_calf_joint",
            "RL_hip_joint",
            "RL_thigh_joint",
            "RL_calf_joint",
        ],
        # PD
        "kp": 20.0,
        "kd": 0.5,
        # termination
        "termination_if_roll_greater_than": 30,  # degree
        "termination_if_pitch_greater_than": 30,
        # base pose
        "base_init_pos": [0.0, 0.0, 0.42],
        "base_init_quat": [1.0, 0.0, 0.0, 0.0],
        "episode_length_s": 20.0,
        "resampling_time_s": 4.0,
        "action_scale": 0.25,
        "simulate_action_latency": True,
        "clip_actions": 100.0,
    }
    obs_cfg = {
        "obs_scales": {
            "lin_vel": 2.0,
            "ang_vel": 0.25,
            "dof_pos": 1.0,
            "dof_vel": 0.05,
        },
    }
    reward_cfg = {
        "tracking_sigma": 0.25,
        "base_height_target": 0.3,
        "feet_height_target": 0.075,
        "reward_scales": {
            "tracking_lin_vel": 1.0,
            "tracking_ang_vel": 0.2,
            "lin_vel_z": -1.0,
            "action_rate": -0.005,
            "similar_to_default": -0.1,
        },
    }
    command_cfg = {
        "num_commands": 3,
        "lin_vel_x_range": LINEAR_VELOCITY_X_RANGE,
        "lin_vel_y_range": LINEAR_VELOCITY_Y_RANGE,
        "ang_vel_range": ANGULAR_VELOCITY_RANGE,
    }

    return env_cfg, obs_cfg, reward_cfg, command_cfg

def save_checkpoints(actor, critic, actor_optim, critic_optim, it):
  torch.save(actor.state_dict(), ACTOR_WEIGHT)
  torch.save(critic.state_dict(), CRITIC_WEIGHT)
  torch.save(actor_optim.state_dict(), ACTOR_OPTIMIZER_WEIGHT)
  torch.save(critic_optim.state_dict(), CRITIC_OPTIMIZER_WEIGHT)
  with open(TRAIN_STATUS, "w") as f:
    json.dump({"iter": it}, f)
  print(f"重みを保存しました (iter:{it})")

def gae(
    rewards, values, dones, gamma, lam
):
  T, N = rewards.shape
  adv = torch.zeros(T,N, device=rewards.device)
  last = 0
  for t in reversed(range(T)):
    nonterminal = 1.0 - dones[t]
    delta = rewards[t] + gamma * values[t+1] * nonterminal - values[t]
    last = delta + gamma * lam * nonterminal * last
    adv[t] = last
  returns = adv + values[:-1]
  adv = (adv - adv.mean()) / (adv.std() + 1e-8)
  return adv, returns

def trainer(
    env, actor, critic, actor_optimizer, critic_optimizer,
    start_iter, num_envs, ppo_cfg, trainer_cfg, save_cfg, iter_upper
):
  from torch.utils.tensorboard import SummaryWriter
  writer = SummaryWriter(log_dir=str(LOGS_DIR))

  iters, steps, num_mini_batches = trainer_cfg["iter"], trainer_cfg["step_per_iter"], trainer_cfg["num_mini_batches"]

  obs = env.reset()
  actor_obs = obs["policy"]
  critic_obs = obs["critic"]

  if iter_upper:
    iters = iter_upper
  for it in range(start_iter, iters):
    actor_obs_buf, critic_obs_buf, actions_buf, logp_buf = [], [], [], []
    values_buf, rewards_buf, dones_buf = [], [], []

    with torch.no_grad():
      for step in range(steps):
        dist = actor(actor_obs)
        actions = dist.sample()
        logp = dist.log_prob(actions).sum(dim=-1)
        values = critic(critic_obs).squeeze(-1)

        obs, rewards, terminated, extras = env.step(actions)
        time_out = extras["time_outs"].bool()

        actor_obs_buf.append(actor_obs)
        critic_obs_buf.append(critic_obs)
        actions_buf.append(actions)
        logp_buf.append(logp)
        values_buf.append(values)
        rewards_buf.append(rewards.clone() + ppo_cfg["gamma"] * values * time_out.float())
        dones_buf.append(terminated.float())

        actor_obs, critic_obs = obs["policy"], obs["critic"]

      values_buf.append(critic(critic_obs).squeeze(-1))

    values = torch.stack(values_buf)
    rewards = torch.stack(rewards_buf)
    dones = torch.stack(dones_buf)
    advantages, returns = gae(rewards, values, dones, ppo_cfg["gamma"], ppo_cfg["lam"])
    advantages = advantages.flatten()
    returns = returns.flatten()
    b_actor_obs = torch.cat(actor_obs_buf)
    b_critic_obs = torch.cat(critic_obs_buf)
    b_actions = torch.cat(actions_buf)
    b_logp = torch.cat(logp_buf)

    B = steps * num_envs
    eps, beta = ppo_cfg["epsilon"], ppo_cfg["entropy_beta"]
    for epoch in range(ppo_cfg["num_epochs"]):
      perm = torch.randperm(B, device=actor_obs.device)
      for idx in perm.chunk(num_mini_batches):
        dist = actor(b_actor_obs[idx])
        logp_new = dist.log_prob(b_actions[idx]).sum(dim=-1)
        ratio = torch.exp(logp_new - b_logp[idx])
        surr = torch.min(ratio * advantages[idx], ratio.clamp(1 - eps, 1 + eps) * advantages[idx])
        entropy = dist.entropy().sum(-1).mean()
        actor_loss = -surr.mean() - beta * entropy

        value = critic(b_critic_obs[idx]).squeeze(-1)
        critic_loss = (returns[idx] - value).pow(2).mean()

        actor_optimizer.zero_grad()
        actor_loss.backward()
        torch.nn.utils.clip_grad_norm_(actor.parameters(), 1.0)
        actor_optimizer.step()

        critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(critic.parameters(), 1.0)
        critic_optimizer.step()

    writer.add_scalar("reward/mean_step_reward", rewards.mean().item(), it)
    writer.add_scalar("loss/actor", actor_loss.item(), it)
    writer.add_scalar("loss/critic", critic_loss.item(), it)
    writer.add_scalar("policy/entropy", entropy.item(), it)
    writer.add_scalar("policy/kl", (b_logp[idx] - logp_new).mean().item(), it)
    writer.add_scalar("policy/clip_fraction", ((ratio - 1).abs() > eps).float().mean().item(), it)
    writer.add_scalar("policy/action_std", dist.stddev.mean().item(), it)
    writer.add_scalar("episode/resets_per_iter", dones.sum().item(), it)

    if (it % save_cfg["save_iter"] == 0 and it != 0):
      save_checkpoints(actor, critic, actor_optimizer, critic_optimizer, it)

  save_checkpoints(actor, critic, actor_optimizer, critic_optimizer, it)
  writer.close()

def main(seed=42, iter_upper=None):
  num_envs, ppo_cfg, trainer_cfg, save_cfg = train_cfgs()
  env_cfg, obs_cfg, reward_cfg, command_cfg = get_cfgs()

  actor = Actor(45,12)
  critic = Critic(52)
  actor_optimizer = torch.optim.Adam(actor.parameters(), lr=trainer_cfg["actor_adam_lr"])
  critic_optimizer = torch.optim.Adam(critic.parameters(), lr=trainer_cfg["critic_adam_lr"])

  if not torch.cuda.is_available():
    print("please enable cuda first")
    return
  device = torch.device("cuda")
  actor.to(device)
  critic.to(device)

  start_iter = 0

  if ACTOR_WEIGHT.exists():
    approved = input(f"{TRAINING_NAME} file already exist. Are you sure to continue?: ")
    if approved.lower() != "y":
      return
    else:
      print("loading weights...")
      actor.load_state_dict(torch.load(ACTOR_WEIGHT))
      critic.load_state_dict(torch.load(CRITIC_WEIGHT))
      actor_optimizer.load_state_dict(torch.load(ACTOR_OPTIMIZER_WEIGHT))
      critic_optimizer.load_state_dict(torch.load(CRITIC_OPTIMIZER_WEIGHT))

      train_status = json.load(open(TRAIN_STATUS))
      start_iter = train_status["iter"] + 1

      print(
          f"分割ファイルから復元完了。次はイテレーション {start_iter} から再開します。"
      )
  else:
    import datetime

    with open(TRAIN_CONFIG, "w") as f:
      train_meta = {
          "name":TRAINING_NAME,
          "method":"ppo",
          "robot":"go2",
          "start_date":datetime.datetime.now().isoformat()
      }
      config_cat = {
          "meta":train_meta,
          "num_envs":num_envs,
          "ppo_cfg":ppo_cfg,
          "trainer_cfg":trainer_cfg,
          "save_cfg":save_cfg,
          "env_cfg":env_cfg,
          "obs_cfg":obs_cfg,
          "reward_cfg":reward_cfg,
          "command_cfg":command_cfg
      }
      json.dump(config_cat, f, indent=2)


  gs.init(backend=gs.gpu, precision="32", logging_level="info", seed=seed, performance_mode=True)

  env = Go2Env(num_envs, env_cfg, obs_cfg, reward_cfg, command_cfg)
  link_names = [link.name for link in env.robot.links]
  print(link_names)
  trainer(env, actor, critic, actor_optimizer, critic_optimizer,
          start_iter, num_envs, ppo_cfg, trainer_cfg, save_cfg, iter_upper)

In [ ]:
# テスト用のイテレーション上限
main()

[Genesis] [04:52:40] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [04:52:40] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [04:52:40] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [04:52:40] [INFO] Running on [NVIDIA A100-SXM4-40GB] with backend gs.cuda. Device memory: 39.49 GB.
[Genesis] [04:52:40] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: 42, 🐛 debug: False, 📏 precision: 32, 🔥 performance: True, 💬 verbose: INFO
[Genesis] [04:52:40] [INFO] Scene <a9cd3c4> created.
[Genesis] [04:52:40] [INFO] Adding <gs.engine.entities.RigidEntity> plane_0f82d34. idx: 0, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/plane/plane.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:52:44] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_93dbd36. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/go2/urdf/go2.u

/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/call_transformer.py:295: DeprecationWarning: keyword.__init__ got an unexpected keyword argument 'ctx'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  kwarg_node = ast.keyword(


Streaming output truncated to the last 5000 lines.
[Genesis] [05:07:43] [INFO] Running at 1,216,297.69 FPS (74.24 FPS per env, 16384 envs).
[Genesis] [05:07:43] [INFO] Running at 1,231,573.83 FPS (75.17 FPS per env, 16384 envs).
[Genesis] [05:07:43] [INFO] Running at 1,246,555.51 FPS (76.08 FPS per env, 16384 envs).
[Genesis] [05:07:43] [INFO] Running at 1,259,346.85 FPS (76.86 FPS per env, 16384 envs).
[Genesis] [05:07:44] [INFO] Running at 1,272,609.64 FPS (77.67 FPS per env, 16384 envs).
[Genesis] [05:07:44] [INFO] Running at 1,215,802.01 FPS (74.21 FPS per env, 16384 envs).
[Genesis] [05:07:44] [INFO] Running at 1,230,358.58 FPS (75.10 FPS per env, 16384 envs).
[Genesis] [05:07:44] [INFO] Running at 1,245,072.37 FPS (75.99 FPS per env, 16384 envs).
[Genesis] [05:07:44] [INFO] Running at 1,258,425.10 FPS (76.81 FPS per env, 16384 envs).
[Genesis] [05:07:44] [INFO] Running at 1,272,005.15 FPS (77.64 FPS per env, 16384 envs).
[Genesis] [05:07:45] [INFO] Running at 1,215,310.83 FPS (74

KeyboardInterrupt: 

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv -l 1

### 不明なクラッシュの原因調査用セル

env作成より先にtensor boardをインポートするとなぜかクラッシュすることがわかった

In [ ]:
%%writefile /content/gs_test.py
import sys
import genesis as gs

mode = sys.argv[1]
gs.init(backend=gs.gpu, logging_level="info")
scene = gs.Scene(show_viewer=False)
scene.add_entity(gs.morphs.Plane())
if mode == "go2":
    scene.add_entity(gs.morphs.URDF(file="urdf/go2/urdf/go2.urdf", pos=(0.0, 0.0, 0.42)))
scene.build(n_envs=64 if mode == "go2" else 1)
print("BUILD OK")
for _ in range(10):
    scene.step()
print("STEP OK")

Writing /content/gs_test.py


In [ ]:
!python -X faulthandler /content/gs_test.py plane; echo "exit code: $?"

[Genesis] [04:13:31] [INFO] ╭─────────╮
[Genesis] [04:13:31] [INFO] │ Genesis │
[Genesis] [04:13:31] [INFO] ╰─────────╯
[Genesis] [04:13:31] [INFO] Running on [NVIDIA A100-SXM4-40GB] with backend gs.cuda. Device memory: 39.49 GB.
[Genesis] [04:13:31] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [04:13:32] [INFO] Scene <2aa4e3c> created.
[Genesis] [04:13:32] [INFO] Adding <gs.engine.entities.RigidEntity> plane_2b0dce0. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [04:13:32] [INFO] Building scene <2aa4e3c>...
[Genesis] [04:13:37] [INFO] Compiling simulation kernels...
/usr/local/lib/python3.13/dist-packages/quadrants/lang/ast/ast_transformers/call_transformer.py:295: DeprecationWarning: keyword.__init__ got an unexpected keyword argument 'ctx'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.
  kwarg_node 

In [ ]:
!python -X faulthandler /content/gs_test.py go2; echo "exit code: $?"

[Genesis] [04:14:25] [INFO] ╭─────────╮
[Genesis] [04:14:25] [INFO] │ Genesis │
[Genesis] [04:14:25] [INFO] ╰─────────╯
[Genesis] [04:14:25] [INFO] Running on [NVIDIA A100-SXM4-40GB] with backend gs.cuda. Device memory: 39.49 GB.
[Genesis] [04:14:25] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [04:14:26] [INFO] Scene <fa42e8d> created.
[Genesis] [04:14:26] [INFO] Adding <gs.engine.entities.RigidEntity> plane_faadf76. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [04:14:29] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_0f11372. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:14:29] [INFO] Building scene <fa42e8d>...
[Genesis] [04:14:30] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [04:1

In [ ]:
%%writefile /content/gs_test2.py
import sys
flags = set(sys.argv[1:])

if "torch" in flags:
    import torch
if "td" in flags:
    from tensordict import TensorDict
if "tb" in flags:
    from torch.utils.tensorboard import SummaryWriter

import genesis as gs

gs.init(backend=gs.gpu, precision="32", logging_level="info", seed=42,
        performance_mode=("perf" in flags))

kwargs = {}
if "sim" in flags:
    kwargs["sim_options"] = gs.options.SimOptions(dt=0.02, substeps=2)
if "rigid" in flags:
    kwargs["rigid_options"] = gs.options.RigidOptions(
        enable_self_collision=False, max_collision_pairs=20, tolerance=1e-5)

scene = gs.Scene(show_viewer=False, **kwargs)
scene.add_entity(gs.morphs.URDF(file="urdf/plane/plane.urdf", fixed=True))
scene.add_entity(gs.morphs.URDF(file="urdf/go2/urdf/go2.urdf", pos=(0, 0, 0.42), quat=(1, 0, 0, 0)))
scene.build(n_envs=64)
print("BUILD OK")
for _ in range(10):
    scene.step()
print("STEP OK")

Writing /content/gs_test2.py


In [ ]:
!python -X faulthandler /content/gs_test2.py perf; echo "exit code: $?"

[Genesis] [04:17:39] [INFO] ╭─────────╮
[Genesis] [04:17:39] [INFO] │ Genesis │
[Genesis] [04:17:39] [INFO] ╰─────────╯
[Genesis] [04:17:39] [INFO] Running on [NVIDIA A100-SXM4-40GB] with backend gs.cuda. Device memory: 39.49 GB.
[Genesis] [04:17:39] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: 42, 🐛 debug: False, 📏 precision: 32, 🔥 performance: True, 💬 verbose: INFO
[Genesis] [04:17:39] [INFO] Scene <fc0f065> created.
[Genesis] [04:17:39] [INFO] Adding <gs.engine.entities.RigidEntity> plane_43c50ee. idx: 0, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/plane/plane.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:17:43] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_7110aef. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:17:43] [INFO] Building scene <fc0f065>...
[Genesis] [04:17:4

In [ ]:
!python -X faulthandler /content/gs_test2.py sim rigid; echo "exit code: $?"


[Genesis] [04:19:33] [INFO] ╭─────────╮
[Genesis] [04:19:33] [INFO] │ Genesis │
[Genesis] [04:19:33] [INFO] ╰─────────╯
[Genesis] [04:19:33] [INFO] Running on [NVIDIA A100-SXM4-40GB] with backend gs.cuda. Device memory: 39.49 GB.
[Genesis] [04:19:33] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: 42, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [04:19:33] [INFO] Scene <0aff669> created.
[Genesis] [04:19:33] [INFO] Adding <gs.engine.entities.RigidEntity> plane_4d3f6e1. idx: 0, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/plane/plane.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:19:37] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_9ec1db7. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:19:37] [INFO] Building scene <0aff669>...
[Genesis] [04:19:

In [ ]:
!python -X faulthandler /content/gs_test2.py torch td tb; echo "exit code: $?"

I0000 00:00:1792988490.728685    9241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792988490.800693    9241 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792988493.221774    9241 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792988493.222321    9241 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [ ]:
!python -X faulthandler /content/gs_test2.py torch; echo "exit code: $?"
!python -X faulthandler /content/gs_test2.py td; echo "exit code: $?"
!python -X faulthandler /content/gs_test2.py tb; echo "exit code: $?"

[Genesis] [04:23:35] [INFO] ╭─────────╮
[Genesis] [04:23:35] [INFO] │ Genesis │
[Genesis] [04:23:35] [INFO] ╰─────────╯
[Genesis] [04:23:35] [INFO] Running on [NVIDIA A100-SXM4-40GB] with backend gs.cuda. Device memory: 39.49 GB.
[Genesis] [04:23:35] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: 42, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO
[Genesis] [04:23:36] [INFO] Scene <2cb494b> created.
[Genesis] [04:23:36] [INFO] Adding <gs.engine.entities.RigidEntity> plane_e4ee25d. idx: 0, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/plane/plane.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:23:39] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_9a1954b. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.13/dist-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [04:23:39] [INFO] Building scene <2cb494b>...
[Genesis] [04:23:

## CaPによる歩行モデルコントロール